# 19-01 · Build an MCP server with FastMCP

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 19 §19.1–§19.4 · type: walkthrough*

**The promise:** by the end you will have a working MCP server whose three tools, one resource, and one prompt are *discovered* from your Python type hints and docstrings — the same artifact a Claude Code session or any MCP host could connect to.

## 🧠 Why this matters

Every tool you have wired so far lived *inside* your agent's codebase. That works until the multiplication bites: N agents × M tools is an integration matrix, and matrices are how platform teams drown.

MCP is **USB-C for AI context**. A capability provider writes one *server*; an AI application writes one *client*; the N×M wiring collapses to N+M. The model never changes — it still just emits tool calls. What MCP standardizes is how capabilities are *discovered, described, and invoked* across hosts.

The payoff you will feel in this notebook: write the server **once**, and the same `search_docs`, `get_ticket`, and `create_ticket` show up in your agent loop (notebook 19-02) and — unchanged — in any other MCP host. That experienced reuse is the whole argument for the protocol (see §19.1).

## Objectives & prerequisites

**By the end you can:**
- expose any capability as an MCP server whose schemas are generated from your function signatures,
- design its tools *for the model* (docstrings as prompts, errors as data),
- enumerate MCP's three primitives — tools, resources, prompts — and *who controls each*.

**Prereqs:** Ch 12 (designing tools for the model) · Ch 13 (`rag_search`, mocked here) · Ch 15 (structured-output discipline).

**Extra package (declared per the standards):** the real server uses the official `mcp` Python SDK (`pip install mcp`). It is **not** required to run this notebook: in `MOCK=1` (the default) we run a tiny in-process stand-in so everything works offline with no install and no API key. Set `MOCK=0` *and* `pip install mcp` to use the genuine `FastMCP`.

## Setup

In [ ]:
import os
import json
import inspect
import random
import typing
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # secrets (if any) come only from .env / the environment

# MOCK=1 (default) runs everything offline with canned, deterministic data.
# MOCK=0 uses the real `mcp` SDK's FastMCP (requires: pip install mcp).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(19)  # determinism for anything stochastic

DATA = Path("data")
DOCS = json.loads((DATA / "docs.json").read_text(encoding="utf-8"))["documents"]
TICKETS = json.loads((DATA / "tickets.json").read_text(encoding="utf-8"))["tickets"]

print(f"MOCK={MOCK} | {len(DOCS)} docs | {len(TICKETS)} seed tickets loaded")
if not MOCK:
    try:
        import mcp  # noqa: F401
    except ImportError:
        raise SystemExit(
            "MOCK=0 needs the real SDK: pip install mcp (or set COMPANION_MOCK=1)."
        )

## The mock backends (Ch 13 RAG + an internal tracker)

An MCP server is a thin, well-described skin over real systems. Here those systems are mocked so every tool answers deterministically offline. `rag_search` returns the closest doc by a trivial keyword overlap (not a real embedding — that is Ch 13's job); the tracker is an in-memory dict.

In [ ]:
def rag_search(query: str, k: int = 5):
    """Mock of the Chapter 13 RAG stack: rank docs by keyword overlap."""
    q = set(query.lower().split())
    scored = []
    for d in DOCS:
        words = set((d["title"] + " " + d["text"]).lower().split())
        scored.append((len(q & words), d))
    scored.sort(key=lambda s: s[0], reverse=True)
    return [d for score, d in scored[:k] if score > 0] or [scored[0][1]]


class Tracker:
    """In-memory stand-in for the internal ticket tracker."""
    def __init__(self, seed):
        self._t = dict(seed)
        self._n = 2000

    def get(self, ticket_id):
        return self._t.get(ticket_id)

    def create(self, title, body, priority):
        self._n += 1
        tid = f"TICK-{self._n}"
        self._t[tid] = {
            "id": tid, "title": title, "body": body,
            "priority": priority, "status": "open", "assignee": None,
        }
        return self._t[tid]


tickets = Tracker(TICKETS)
print("backends ready:", rag_search("refund")[0]["doc_id"])

## The three primitives — and *who controls use*

Before any code, the distinction that organizes the whole protocol (§19.2):

| Primitive | What it is | Controlled by |
|---|---|---|
| **Tools** | Executable actions with JSON-Schema inputs | The **model** (the agent decides to call) |
| **Resources** | Readable context addressed by a URI | The **application** (host attaches it to context) |
| **Prompts** | Reusable, parameterized prompt templates | The **user** (explicitly invoked) |

Most servers lead with tools, but resources and prompts let *one* server feed every layer of an agent's context, not just its actions.

## 🔧 Build the server (§19.4)

Here is the real thing, exactly as it appears in the book. `FastMCP` makes the server *declarative*: the type hints become the input schema, and the docstring becomes the tool's description that the model reads.

We define it once, in a way that works **both** with the genuine SDK (`MOCK=0`) and with our in-process stand-in (`MOCK=1`). The decorated functions are byte-for-byte the book's — only the `FastMCP` object differs.

In [ ]:
# A minimal, in-process stand-in for mcp.server.fastmcp.FastMCP.
# It captures exactly what a real client would *discover*: a tool's name,
# its docstring (the model-facing description), and a JSON Schema built
# from the signature. It does NOT reimplement the wire protocol.

_PY_TO_JSON = {str: "string", int: "integer", float: "number", bool: "boolean"}


def _schema_from_signature(fn):
    props, required = {}, []
    for name, p in inspect.signature(fn).parameters.items():
        ann = p.annotation if p.annotation is not inspect._empty else str
        props[name] = {"type": _PY_TO_JSON.get(ann, "string")}
        if p.default is inspect._empty:
            required.append(name)
        else:
            props[name]["default"] = p.default
    return {"type": "object", "properties": props, "required": required}


class MockFastMCP:
    def __init__(self, name):
        self.name = name
        self.tools, self.resources, self.prompts = {}, {}, {}

    def tool(self):
        def deco(fn):
            self.tools[fn.__name__] = {
                "fn": fn,
                "description": inspect.getdoc(fn) or "",
                "inputSchema": _schema_from_signature(fn),
            }
            return fn
        return deco

    def resource(self, uri_template):
        def deco(fn):
            self.resources[uri_template] = {
                "fn": fn, "description": inspect.getdoc(fn) or ""}
            return fn
        return deco

    def prompt(self):
        def deco(fn):
            self.prompts[fn.__name__] = {
                "fn": fn, "description": inspect.getdoc(fn) or ""}
            return fn
        return deco

    def run(self, transport="stdio"):  # pragma: no cover - not used in mock
        raise RuntimeError("mock server runs in-process; use call_* helpers")


if MOCK:
    FastMCP = MockFastMCP
else:
    from mcp.server.fastmcp import FastMCP  # the genuine SDK

RUNBOOK_DIR = DATA
print("FastMCP =", FastMCP.__name__)

In [ ]:
# capstone-project/mcp_server.py  (the book's §19.4 server, verbatim shapes)
mcp = FastMCP("capstone-enterprise")


@mcp.tool()
def search_docs(query: str, k: int = 5) -> str:
    """Search the company document base when you need policy, SLA, or
    known-issue context. Returns the top passages, each prefixed with its
    source document id. Use this before answering factual questions."""
    hits = rag_search(query, k=k)
    return "\n\n".join(f"[{h['doc_id']}] {h['text']}" for h in hits)


@mcp.tool()
def get_ticket(ticket_id: str) -> str:
    """Fetch one support ticket (status, assignee, priority, history) by id.
    Use when the user references a ticket or you must inspect its state."""
    t = tickets.get(ticket_id)
    if t is None:
        return f"error: no ticket {ticket_id}"  # errors as data, not crashes
    return json.dumps(t)


@mcp.tool()
def create_ticket(title: str, body: str, priority: str = "normal") -> str:
    """Create a support ticket and return its new id. Call this only to log a
    genuinely new issue. Priority must be one of: low, normal, high."""
    if priority not in ("low", "normal", "high"):
        return "error: invalid priority"  # errors as data, not crashes
    return tickets.create(title=title, body=body, priority=priority)["id"]


@mcp.resource("runbook://{name}")
def runbook(name: str) -> str:
    """Operational runbooks, addressable by name."""
    return (RUNBOOK_DIR / f"runbook-{name}.md").read_text(encoding="utf-8")


@mcp.prompt()
def triage(ticket_id: str) -> str:
    """Standard triage workflow for a support ticket."""
    return (f"Triage ticket {ticket_id}: fetch it with get_ticket, "
            "search_docs for related known issues, then propose "
            "severity, owner, and first response.")


print("server:", mcp.name)

## ⚠️ Pitfall: tools written for the *API*, not the *model*

The most common MCP mistake is to expose your internal function signatures unchanged — `search_docs(q, n)` with the docstring *"runs a vector query"*. The model has no idea **when** to reach for it.

Notice the docstrings above are written as **prompts**: they say *when to use the tool*, not just what it does ("Use this before answering factual questions"). And every failure path returns a clean string — `"error: invalid priority"`, `"error: no ticket TICK-9"` — never a raised exception. A clean message is something the agent can recover from; a stack trace is noise that derails the loop (§19.4 tip).

## 🔮 Predict, then run

A client is about to connect and ask the server *what do you offer?* — the discovery handshake. **Before running the next cell, predict:**

1. How many tools will it discover, and where do their input schemas come from?
2. What will `create_ticket`'s schema say about the `priority` argument's default?
3. Will the `runbook` resource and the `triage` prompt show up in the *tools* list?

Write your guesses down, then run.

In [ ]:
# An MCP-Inspector-style discovery, run in-process. With the real SDK the
# same information arrives over JSON-RPC; the *shape* is identical.
def inspect_server(server):
    print(f"# server: {server.name}\n")
    print(f"TOOLS ({len(server.tools)}):")
    for name, spec in server.tools.items():
        params = ", ".join(spec["inputSchema"]["properties"])
        print(f"  - {name}({params})")
        print(f"      desc: {spec['description'].splitlines()[0]}")
    print(f"\nRESOURCE TEMPLATES ({len(server.resources)}):")
    for uri in server.resources:
        print(f"  - {uri}")
    print(f"\nPROMPTS ({len(server.prompts)}):")
    for name in server.prompts:
        print(f"  - {name}")


if MOCK:
    inspect_server(mcp)
else:
    print("Run the real server under the MCP Inspector: `mcp dev capstone-project/mcp_server.py`")

In [ ]:
# The single most important artifact: the JSON Schema the model actually sees,
# generated entirely from create_ticket's signature + docstring.
if MOCK:
    spec = mcp.tools["create_ticket"]
    print(json.dumps({
        "name": "create_ticket",
        "description": spec["description"],
        "inputSchema": spec["inputSchema"],
    }, indent=2))

**What you just saw.** Three tools, each with a schema *derived from the signature* — `priority` carries its `"normal"` default straight from the Python default. The resource and the prompt are listed **separately**, because they are controlled by the application and the user respectively, not by the model. You wrote functions; the protocol turned them into a discoverable contract.

## Calling the capabilities

Discovery is half the protocol; the other half is invocation. In `MOCK=1` we dispatch in-process; in 19-02 the same calls go out over a real client session. Watch the *errors-as-data* contract hold.

In [ ]:
def call_tool(server, name, args):
    return server.tools[name]["fn"](**args)


def read_resource(server, name):
    return server.resources["runbook://{name}"]["fn"](name)


if MOCK:
    print("search_docs ->")
    print(call_tool(mcp, "search_docs", {"query": "refund policy", "k": 2}))
    print("\nget_ticket(TICK-1001) ->")
    print(call_tool(mcp, "get_ticket", {"ticket_id": "TICK-1001"}))
    print("\ncreate_ticket(bad priority) ->",
          call_tool(mcp, "create_ticket",
                    {"title": "x", "body": "y", "priority": "URGENT"}))
    print("create_ticket(valid) ->",
          call_tool(mcp, "create_ticket",
                    {"title": "SSO loop", "body": "repro attached", "priority": "high"}))
    print("\nrunbook://sso-outage (first line) ->")
    print(read_resource(mcp, "sso-outage").splitlines()[0])

## Transports: local now, networked later

We ran in-process for clarity. Two real transports matter (§19.2):

- **stdio** — the server is a child process; the host talks to it over stdin/stdout. This is the default for local development: `mcp.run()`.
- **streamable HTTP** — the server is a web service: `mcp.run(transport="streamable-http")`. This is the production path, deployed behind the Part VII FastAPI gateway **with auth** — covered in 19-02 and the blueprint.

The decorated tool functions do not change between transports. That invariance is the point.

## 🎯 Senior lens

Scope **one server per capability *domain***, not per endpoint — a `docs` server, a `tickets` server, a `repo` server — each **owned, deployed, and audited by the team that owns the underlying system**. Once you do that, tools stop being code compiled into an agent and become *infrastructure* with their own release cycle, access controls, and on-call. The org chart, not the framework, ends up shaping where the boundaries fall (§19.4) — which is exactly what you want, because those boundaries outlive any agent you write against them.

## Recap

- MCP is USB-C for AI context: one server, one client, N×M → N+M; the model is unchanged.
- A server exposes **tools** (model-controlled), **resources** (app-controlled), **prompts** (user-controlled).
- `FastMCP` is *declarative*: type hints → input schema, docstring → the model-facing description.
- Design tools **for the model** — docstrings as prompts, and **errors returned as data**, not raised.
- One server per capability domain, owned by the team that owns the system.

## Exercises

Each exercise *changes* something and asks you to predict the result first.

1. **A fourth tool.** Add `close_ticket(ticket_id: str) -> str` that flips a ticket's status to `closed` and returns a confirmation, or `"error: no ticket …"` for an unknown id. *Predict its discovered schema before you re-run the inspector.*
2. **Tighten a contract.** Change `search_docs`'s `k` default to `3` and add a sentence to its docstring telling the model to prefer a small `k`. Re-inspect: which fields in the JSON Schema changed, and which did not?
3. **A second resource.** Add a `policy://{name}` resource that reads a doc from `data/docs.json` by `doc_id`. Predict whether it appears under TOOLS or RESOURCES, and why.
4. **Break it on purpose.** Give `get_ticket` a parameter with **no** type annotation and re-inspect. What type does the generated schema fall back to, and why is an explicit annotation the safer habit?

In [ ]:
# Exercise 1 — close_ticket


In [ ]:
# Exercise 2 — change the search_docs default + docstring, then re-inspect


In [ ]:
# Exercise 3 — a policy:// resource


In [ ]:
# Exercise 4 — drop an annotation and observe the schema fallback


## Next

- ➡️ **Next notebook:** [`19-02-consume-mcp-and-security.ipynb`](19-02-consume-mcp-and-security.ipynb) — bridge these discovered tools into the Ch 12 agent loop and draw the three security boundaries.
- 🧩 **Blueprint (the real thing):** [`../../blueprints/mcp-server/`](../../blueprints/mcp-server/) — you built the toy; this is the production server with real transports, OAuth on streamable HTTP, per-call audit logging, and isolation tests.
- 🏗️ **Capstone:** this server becomes [`../../capstone-project/mcp/`](../../capstone-project/mcp/) (checkpoint `checkpoints/ch19-mcp-server`), deployed behind the Part VII FastAPI gateway.